# B3 — H1 semi-norm re-evaluation of every existing checkpoint

**Closing a real gap found while reviewing the investigation report against Professor Rabczuk's own emails**: his 2026-10-20 email explicitly asked to reduce errors in "all stress components, the H1 semi-norm, and other local QoIs" -- but the H1 semi-norm itself was never actually computed for any of the four candidate error-reduction methods tried so far.

**This is a CHEAP notebook, not a training run**: `compute_H1_seminorm_rel` (new, in `evaluate_B3_qois.py`, CPU-unit-tested against a closed-form affine-field reference to machine precision) is now wired into `evaluate_B3_qois.py`, `evaluate_B3_mixed_stress.py`, and a new `reevaluate_B3_data_driven.py`. This notebook just RE-RUNS evaluation (no retraining) on the 7 checkpoints that already exist on Drive from previous runs, against the same held-out FEM set each was already evaluated on:

1. PI baseline (`b3_training_normalized`, 1.88% combined displacement error)
2. Item 1, attempt 1 (mixed-stress, `P_OUTPUT_SCALE=500`)
3. Item 1, attempt 4 (mixed-stress, relative constitutive residual)
4. Items 2/3, attempt 1 (equilibrium-augmented, `w_equilibrium=1.0`)
5. Items 2/3, attempt 2 (equilibrium-augmented, `w_equilibrium=0.12`)
6. Item 4, data-driven `rel_l2`
7. Item 4, data-driven `sobolev`

Each re-evaluation is a handful of forward passes over 100 held-out samples -- expect this to finish in a few minutes total, nothing like a 50,000-iteration training run.

In [ ]:
# =====================================================================
#  CELL -- Re-evaluate all 7 existing B3 checkpoints with the new H1
#  semi-norm metric added. No training -- just loading each checkpoint
#  and running it against the same held-out FEM set it was already
#  evaluated on.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU is fine for this -- just forward passes)')

print(f'\n{"=" * 90}')
print('=== Re-running the new H1 semi-norm correctness checks in this environment ===')
from omar_pfem.test_evaluate_B3_h1_seminorm import (
    test_zero_for_identical_fields, test_matches_hand_computation_for_affine_field,
    test_nonzero_and_finite_for_random_fields,
)
test_zero_for_identical_fields()
test_matches_hand_computation_for_affine_field()
test_nonzero_and_finite_for_random_fields()

R = '/content/drive/MyDrive/pfem_run'
HOLDOUT = f'{R}/b3_dataset_clean_holdout/dataset.h5'

JOBS = [
    ('PI baseline', 'evaluate_B3_qois', f'{R}/b3_training_normalized/checkpoint_50000.pt'),
    ('Item 1, attempt 1 (mixed-stress, scale=500)', 'evaluate_B3_mixed_stress',
     f'{R}/b3_mixed_stress_training/checkpoint_50000.pt'),
    ('Item 1, attempt 4 (mixed-stress, relative)', 'evaluate_B3_mixed_stress',
     f'{R}/b3_mixed_stress_relative/checkpoint_50000.pt'),
    ('Items 2/3, attempt 1 (equilibrium, w=1.0)', 'evaluate_B3_qois',
     f'{R}/b3_equilibrium_augmented/checkpoint_50000.pt'),
    ('Items 2/3, attempt 2 (equilibrium, w=0.12)', 'evaluate_B3_qois',
     f'{R}/b3_equilibrium_augmented_v2/checkpoint_50000.pt'),
]

h1_summary = []
for name, module, ckpt in JOBS:
    print(f'\n{"=" * 90}')
    print(f'=== {name} ===')
    if not os.path.exists(ckpt):
        print(f'  SKIPPED -- checkpoint not found at {ckpt}')
        continue
    out_json = ckpt.replace('.pt', '_h1_reeval.json')
    sys.argv = [f'{module}.py', '--checkpoint', ckpt, '--dataset', HOLDOUT, '--out_json', out_json]
    if module == 'evaluate_B3_qois':
        from omar_pfem.evaluate_B3_qois import main as eval_main
    else:
        from omar_pfem.evaluate_B3_mixed_stress import main as eval_main
    eval_main()
    import json as _json
    with open(out_json) as f:
        res = _json.load(f)
    h1_summary.append((name, res['summary'].get('mean_H1_seminorm_rel'), res['summary'].get('median_H1_seminorm_rel')))

# Item 4's two data-driven checkpoints use the separate reevaluate_B3_data_driven.py
# (no training dataset needed, unlike train_B3_data_driven.py's own main()).
for name, loss in [('Item 4, data-driven rel_l2', 'rel_l2'), ('Item 4, data-driven sobolev', 'sobolev')]:
    print(f'\n{"=" * 90}')
    print(f'=== {name} ===')
    ckpt = f'{R}/b3_data_driven_{loss}/model_best.pt'
    if not os.path.exists(ckpt):
        print(f'  SKIPPED -- checkpoint not found at {ckpt}')
        continue
    out_json = f'{R}/b3_data_driven_{loss}/qois_h1_reeval.json'
    sys.argv = ['reevaluate_B3_data_driven.py', '--checkpoint', ckpt, '--test_dataset', HOLDOUT,
                '--out_json', out_json]
    from omar_pfem.reevaluate_B3_data_driven import main as reeval_main
    reeval_main()
    import json as _json
    with open(out_json) as f:
        res = _json.load(f)
    h1_summary.append((name, res['metrics']['mean_H1_seminorm_rel'], res['metrics']['median_H1_seminorm_rel']))

print(f'\n{"=" * 90}')
print('=== H1 semi-norm summary across all checkpoints ===')
for name, mean_h1, median_h1 in h1_summary:
    print(f'  {name}: mean H1_seminorm_rel={mean_h1}, median={median_h1}')
print(f'\nTotal wall-clock: {time.time() - _started:.1f}s')
print('\nDone. Send this cell\'s full printed output back.')
